In [2]:
import hashlib
import os
import platform
import uuid
# 修改为自己的学号
STUDENT_ID = 20244080226
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] hw01_env
[学号]       20244080226
[机器指纹] 068b3410f591ad41
[RUN_ID]   20244080226-068b3410f591ad41


## Python 正则表达式练习

In [3]:
import re
def check_alnum(s):
    return bool(re.fullmatch(r'[a-zA-Z0-9]+', s))
# 测试
print(check_alnum("Hello123")) # True
print(check_alnum("Hello 123")) # False (有空格)

True
False


In [4]:
def match_a_zero_or_more_b(s):
    return bool(re.fullmatch(r'ab*', s))
# 测试
print(match_a_zero_or_more_b("a")) # True
print(match_a_zero_or_more_b("abbb")) # True
print(match_a_zero_or_more_b("ac")) # False

True
True
False


In [5]:
def match_a_one_or_more_b(s):
    return bool(re.fullmatch(r'ab+', s))
# 测试
print(match_a_one_or_more_b("ab")) # True
print(match_a_one_or_more_b("a")) # False

True
False


In [6]:
def match_a_to_b(s):
    # .* 表示任意字符出现任意次（包括0次）
    return bool(re.fullmatch(r'a.*b', s))
# 测试
print(match_a_to_b("ab")) # True
print(match_a_to_b("a123b")) # True
print(match_a_to_b("a123c")) # False

True
True
False


In [7]:
def remove_ip_leading_zeros(ip):
    # \b0+(?=\d) 匹配：位于单词边界，紧跟至少一个0，且0后面还有数字
    return re.sub(r'\b0+(?=\d)', '', ip)
# 测试
print(remove_ip_leading_zeros("192.168.001.010")) # 192.168.1.10
print(remove_ip_leading_zeros("192.168.000.000")) # 192.168.0.0

192.168.1.10
192.168.0.0


In [8]:
def swap_space_underscore(s):
    # 巧妙利用替换函数，遇到空格换下划线，遇到下划线换空格
    return re.sub(r'[ _]', lambda x: '_' if x.group() == ' ' else ' ', s)
# 测试
print(swap_space_underscore("hello world_test")) # hello_world test

hello_world test


In [9]:
def convert_date(date):
    # 利用捕获组 (\d{4}) 捕获年份，\1 \2 \3 代表第一、二、三个括号里的内容
    return re.sub(r'(\d{4})-(\d{2})-(\d{2})', r'\3-\2-\1', date)
# 测试
print(convert_date("2023-10-25")) # 25-10-2023

25-10-2023


In [13]:
def abbreviate_road(s):
    return re.sub(r'\bRoad\b', 'Rd', s)
# 测试
print(abbreviate_road("Main Road and Crossroad")) # Main Rd and Crossroad (Crossroad不会被误伤)

Main Rd and Crossroad


In [14]:
def find_words_starting_ae(s):
    # \b 表示单词边界，[ae]表示首字母，\w*表示后续任意字母数字
    return re.findall(r'\b[ae]\w*', s, re.IGNORECASE)
# 测试
print(find_words_starting_ae("An apple a day keeps the doctor away")) # ['An', 'apple', 'a', 'away']

['An', 'apple', 'a', 'away']


In [15]:
def remove_special_chars(s):
    # ^ 在方括号内表示“非”
    return re.sub(r'[^a-zA-Z0-9]', '', s)
# 测试
print(remove_special_chars("Hello, World! 123.")) # HelloWorld123

HelloWorld123


In [16]:
def remove_brackets_content(text):
    # \( 和 \) 转义括号，.*? 匹配括号内任意非贪婪字符
    # 针对作业里的多行文本，逐行处理
    lines = text.strip().split('\n')
    return [re.sub(r'\(.*?\)', '', line).strip() for line in lines]
# 测试
text = """example (.com)
w3resource
github (.com)
stackoverflow (.com)"""
print(remove_brackets_content(text)) # ['example', 'w3resource', 'github', 'stackoverflow']

['example', 'w3resource', 'github', 'stackoverflow']


## 从网页检索气候数据

In [32]:
import re
from bs4 import BeautifulSoup

file_path = "哈尔滨市_百度百科.html"

# 读取解析
with open(file_path, 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

In [33]:
def retrieve_climate_table(s):
    """接收文件路径，返回气候表格的 BeautifulSoup Tag 对象"""
    with open(s, 'r', encoding='utf-8') as f:
        soup_local = BeautifulSoup(f.read(), 'html.parser')
    
    # 遍历所有 table，不限制 class
    for table in soup_local.find_all('table'):
        text = table.get_text()
        # 根据你截图里的特征：包含“气象数据”且包含“降水量”
        if "气象数据" in text and "降水量" in text:
            return table
    return None

table_obj = retrieve_climate_table(file_path)
print("返回对象类型:", type(table_obj))

返回对象类型: <class 'bs4.element.Tag'>


In [34]:
def list_climate_table_row_names(s):
    """接收文件路径，返回其行名组成的列表"""
    table = retrieve_climate_table(s)
    if table is None:
        return None
        
    row_names = []
    # 遍历表格的每一行
    for tr in table.find_all('tr'):
        # 找到该行的所有单元格
        cells = tr.find_all(['th', 'td'])
        if not cells:
            continue
            
        # 获取第一列的文本
        first_cell = cells[0]
        name = first_cell.get_text(strip=True)
        
        # 清理可能存在的引用角标 [1]、[2] 以及多余空格
        name = re.sub(r'\[\d+\]', '', name).strip()
        
        # 过滤：排除空字符、表头行（包含“月份”、“气象数据”等）
        if name and "月份" not in name and "气象数据" not in name and "数据来源" not in name:
            # 有时候百度会把一些数据脚注也弄进来，过滤掉纯数字
            if not name.isdigit():
                row_names.append(name)
            
    return row_names

print("提取的行名列表：")
row_names = list_climate_table_row_names(file_path)
if row_names:
    for name in row_names:
        print(f" - {name}")
else:
    print("未提取到行名。")

提取的行名列表：
 - 平均最高温度（°C）
 - 平均最低温度（°C）
 - 平均降水量（mm）
